# SuperAI Season 6 - Level Individual Hackathon - Heart Disease Prediction

Pipeline: **LightGBM + XGBoost + CatBoost** (5-fold CV) → ensemble → **F2 threshold tuning**

> **Metric ของ competition = F2 score** (β=2 — เน้น recall / จับ positive ให้ครบ) ดังนั้น threshold ถูกจูนบน F2 ไม่ใช่ F1

**Flow:** Setup Data → Data Exploration (EDA) → Preprocessing & Feature Engineering → CV Training (3 โมเดล) → Evaluation → Submission

> รันบน Colab ได้เลย (เป็น tabular ไม่ต้องใช้ GPU ก็ได้ แต่มีก็เร็วขึ้น)

## Setup Data

In [1]:
!pip install -q lightgbm xgboost catboost scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Setup Data — ดาวน์โหลดจาก Kaggle
# ============================================================
import os

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    print('Upload your kaggle.json:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

COMP_NAME = 'super-ai-engineer-ss-6-heart-disease-prediction'
if not os.path.exists('dataset'):
    !kaggle competitions download -c {COMP_NAME}
    !mkdir -p dataset && unzip -qo {COMP_NAME}.zip -d dataset/
    print('Data downloaded:', os.listdir('dataset'))
else:
    print('Data exists:', os.listdir('dataset'))

Upload your kaggle.json:


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, fbeta_score, classification_report,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)

def f2_score(y_true, y_pred):
    """F2 = fbeta with beta=2 (competition metric)"""
    return fbeta_score(y_true, y_pred, beta=2)

### Load Data

In [ ]:
# รองรับทั้งกรณี unzip แล้วไฟล์อยู่ใต้ dataset/ ตรง ๆ หรือซ้อนโฟลเดอร์
import glob
def find_csv(name):
    hits = glob.glob(f'dataset/**/{name}', recursive=True) + glob.glob(name)
    return hits[0] if hits else f'dataset/{name}'

train = pd.read_csv(find_csv('train.csv'))
test  = pd.read_csv(find_csv('test.csv'))
sub   = pd.read_csv(find_csv('sample_submission.csv'))

TARGET = 'History of HeartDisease or Attack'
print(f'Train: {train.shape}, Test: {test.shape}')
train.head()

## Data Exploration (EDA)

### Target distribution (imbalanced)

In [ ]:
print('=== Target Distribution ===')
print(train[TARGET].value_counts(dropna=False))
print(f'\nPositive rate: {(train[TARGET]=="Yes").mean():.2%}')

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
train[TARGET].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('Target Distribution (Imbalanced)'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

### Missing values

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('=== Missing Values ===')
for col, cnt in missing.items():
    print(f'  {col}: {cnt:,} ({cnt/len(train):.1%})')
print(f'\nTest missing: {test.isnull().sum().sum()}')

### Numeric features (Age, BMI) by target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(['Age', 'Body Mass Index']):
    for label, color in [('No', 'steelblue'), ('Yes', 'salmon')]:
        subset = train[train[TARGET] == label][col].dropna()
        axes[i].hist(subset, bins=50, alpha=0.5, label=label, color=color, density=True)
    axes[i].set_title(f'{col} by Target'); axes[i].legend()
plt.tight_layout(); plt.show()

### Categorical features vs heart disease rate

In [ ]:
cat_cols = ['High Blood Pressure', 'Told High Cholesterol', 'Smoked 100+ Cigarettes',
            'Diagnosed Stroke', 'Diagnosed Diabetes', 'Difficulty Walking',
            'General Health', 'Sex']
fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(train[col], train[TARGET], normalize='index')
    if 'Yes' in ct.columns:
        ct['Yes'].sort_values().plot(kind='barh', ax=axes[i], color='salmon')
    axes[i].set_title(col); axes[i].set_xlabel('Heart Disease Rate')
plt.tight_layout(); plt.show()

### Heart disease rate by age group

In [ ]:
train_clean = train.dropna(subset=[TARGET]).copy()
train_clean['Age_bin'] = pd.cut(train_clean['Age'], bins=range(15, 105, 5))
age_rate = train_clean.groupby('Age_bin')[TARGET].apply(lambda x: (x == 'Yes').mean())
fig, ax = plt.subplots(figsize=(10, 4))
age_rate.plot(kind='bar', ax=ax, color='salmon')
ax.set_title('Heart Disease Rate by Age Group'); ax.set_ylabel('Rate')
plt.tight_layout(); plt.show()

### Correlation of risk factors with target

In [ ]:
temp = train.dropna(subset=[TARGET]).copy()
temp[TARGET] = (temp[TARGET] == 'Yes').astype(int)
binary_features = ['High Blood Pressure', 'Told High Cholesterol', 'Smoked 100+ Cigarettes',
                   'Diagnosed Stroke', 'Diagnosed Diabetes', 'Difficulty Walking',
                   'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Sex']
for col in binary_features:
    temp[col] = temp[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})
corr_cols = binary_features + ['Age', 'Body Mass Index', TARGET]
corr = temp[corr_cols].corr()[TARGET].drop(TARGET).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind='barh', ax=ax, color=['salmon' if v > 0 else 'steelblue' for v in corr])
ax.set_title('Correlation with Heart Disease'); ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()

## Preprocessing & Feature Engineering

In [ ]:
# Drop rows with missing target
train = train.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'Train after dropping missing target: {train.shape}')
train[TARGET] = (train[TARGET] == 'Yes').astype(int)

# บาง submission มีคำตอบบางส่วนอยู่แล้ว — เติมเฉพาะที่ยังว่าง
known_mask = sub[TARGET].notna()
predict_mask = sub[TARGET].isna()
print(f'Known answers in submission: {known_mask.sum()}, To predict: {predict_mask.sum()}')

In [ ]:
binary_map = {'Yes': 1, 'No': 0}
sex_map = {'Male': 1, 'Female': 0}
general_health_map = {'Very Poor': 0, 'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4}
education_map = {'Never attended school': 0, 'Elementary': 1, 'Some high school': 2,
                 'High school graduate': 3, 'Some college or technical school': 4, 'College graduate': 5}
income_map = {'Less than $10,000': 0, '($10,000 to less than $15,000': 1,
              '$15,000 to less than $20,000': 2, '$20,000 to less than $25,000': 3,
              '$25,000 to less than $35,000': 4, '$35,000 to less than $50,000': 5,
              '$50,000 to less than $75,000': 6, '$75,000 or more': 7}
binary_cols = ['High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked',
               'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes',
               'Leisure Physical Activity', 'Heavy Alcohol Consumption',
               'Health Care Coverage', 'Doctor Visit Cost Barrier',
               'Difficulty Walking', 'Vegetable or Fruit Intake (1+ per Day)']

In [ ]:
def feature_engineering(df):
    df = df.copy()
    # --- Base encoding ---
    for col in binary_cols: df[col] = df[col].map(binary_map)
    df['Sex'] = df['Sex'].map(sex_map)
    df['General Health'] = df['General Health'].map(general_health_map)
    df['Education Level'] = df['Education Level'].map(education_map)
    df['Income Level'] = df['Income Level'].map(income_map)
    # --- BMI categories (WHO) ---
    df['BMI_underweight']  = (df['Body Mass Index'] < 18.5).astype(int)
    df['BMI_overweight']   = (df['Body Mass Index'] >= 25).astype(int)
    df['BMI_obese']        = (df['Body Mass Index'] >= 30).astype(int)
    df['BMI_severe_obese'] = (df['Body Mass Index'] >= 40).astype(int)
    # --- Age groups ---
    df['Age_senior']  = (df['Age'] >= 65).astype(int)
    df['Age_elderly'] = (df['Age'] >= 75).astype(int)
    # --- Risk factor count ---
    risk_cols = ['High Blood Pressure', 'Told High Cholesterol', 'Smoked 100+ Cigarettes',
                 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Difficulty Walking']
    df['risk_factor_count'] = df[risk_cols].sum(axis=1)
    df['high_bp_cholesterol'] = (df['High Blood Pressure'] + df['Told High Cholesterol'] == 2).astype(int)
    df['stroke_diabetes'] = (df['Diagnosed Stroke'] + df['Diagnosed Diabetes'] >= 1).astype(int)
    # --- Lifestyle / access / interactions ---
    df['lifestyle_score'] = (df['Leisure Physical Activity'] +
        df['Vegetable or Fruit Intake (1+ per Day)'] +
        (1 - df['Smoked 100+ Cigarettes'].fillna(0)) + (1 - df['Heavy Alcohol Consumption']))
    df['health_access'] = df['Health Care Coverage'] + (1 - df['Doctor Visit Cost Barrier'].fillna(0))
    df['age_bmi']  = df['Age'] * df['Body Mass Index']
    df['age_risk'] = df['Age'] * df['risk_factor_count']
    df['poor_health_senior'] = ((df['General Health'] <= 1) & (df['Age'] >= 60)).astype(int)
    # --- Drop ID / target ---
    df = df.drop(columns=['ID'], errors='ignore')
    if TARGET in df.columns: df = df.drop(columns=[TARGET])
    return df

X = feature_engineering(train)
y = train[TARGET].values
X_test = feature_engineering(test)
print(f'Features: {X.shape[1]} | X: {X.shape}, X_test: {X_test.shape}')
X.head()

## Model Training & Inference — 5-Fold CV (LightGBM + XGBoost + CatBoost)

In [ ]:
N_FOLDS = 5; SEED = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

### LightGBM

In [ ]:
lgb_oof = np.zeros(len(X)); lgb_test = np.zeros(len(X_test))
lgb_params = {'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': -1, 'min_child_samples': 50,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'scale_pos_weight': (y == 0).sum() / (y == 1).sum(), 'verbose': -1, 'seed': SEED}
for fold, (tr, va) in enumerate(skf.split(X, y)):
    model = lgb.train(lgb_params, lgb.Dataset(X.iloc[tr], y[tr]), num_boost_round=2000,
        valid_sets=[lgb.Dataset(X.iloc[va], y[va])],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    lgb_oof[va] = model.predict(X.iloc[va])
    lgb_test += model.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1} F2: {f2_score(y[va], (lgb_oof[va]>0.5).astype(int)):.4f}')
print(f'\nLightGBM OOF F2: {f2_score(y, (lgb_oof>0.5).astype(int)):.4f}')

### XGBoost

In [ ]:
xgb_oof = np.zeros(len(X)); xgb_test = np.zeros(len(X_test))
xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'logloss',
    'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 50,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),
    'seed': SEED, 'tree_method': 'hist', 'verbosity': 0}
for fold, (tr, va) in enumerate(skf.split(X, y)):
    dval = xgb.DMatrix(X.iloc[va], y[va])
    model = xgb.train(xgb_params, xgb.DMatrix(X.iloc[tr], y[tr]), num_boost_round=2000,
        evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=500)
    xgb_oof[va] = model.predict(dval)
    xgb_test += model.predict(xgb.DMatrix(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} F2: {f2_score(y[va], (xgb_oof[va]>0.5).astype(int)):.4f}')
print(f'\nXGBoost OOF F2: {f2_score(y, (xgb_oof>0.5).astype(int)):.4f}')

### CatBoost

In [ ]:
cat_oof = np.zeros(len(X)); cat_test = np.zeros(len(X_test))
for fold, (tr, va) in enumerate(skf.split(X, y)):
    model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
        auto_class_weights='Balanced', eval_metric='Logloss',
        random_seed=SEED, verbose=500, early_stopping_rounds=100)
    model.fit(X.iloc[tr], y[tr], eval_set=(X.iloc[va], y[va]))
    cat_oof[va] = model.predict_proba(X.iloc[va])[:, 1]
    cat_test += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold+1} F2: {f2_score(y[va], (cat_oof[va]>0.5).astype(int)):.4f}')
print(f'\nCatBoost OOF F2: {f2_score(y, (cat_oof>0.5).astype(int)):.4f}')

## Evaluation

### Feature importance (LightGBM, gain)

In [ ]:
lgb_full = lgb.train(lgb_params, lgb.Dataset(X, y), num_boost_round=500)
imp = pd.DataFrame({'feature': X.columns,
    'importance': lgb_full.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(8, 8))
imp.plot(kind='barh', x='feature', y='importance', ax=ax, legend=False, color='steelblue')
ax.set_title('LightGBM Feature Importance (gain)'); plt.tight_layout(); plt.show()

### Ensemble & F2 threshold tuning

In [ ]:
ens_oof  = (lgb_oof + xgb_oof + cat_oof) / 3
ens_test = (lgb_test + xgb_test + cat_test) / 3

# จูน threshold บน F2 (metric จริงของ competition)
best_thr, best_f2, results = 0.5, 0, []
for thr in np.arange(0.15, 0.65, 0.005):
    f2 = f2_score(y, (ens_oof > thr).astype(int))
    results.append((thr, f2))
    if f2 > best_f2: best_f2, best_thr = f2, thr

res_df = pd.DataFrame(results, columns=['threshold', 'f2'])
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(res_df['threshold'], res_df['f2'])
ax.axvline(x=best_thr, color='red', linestyle='--', label=f'Best: {best_thr:.3f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('F2'); ax.set_title('F2 Threshold Tuning'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
print('--- Model Comparison (OOF) ---')
for name, oof in [('LightGBM', lgb_oof), ('XGBoost', xgb_oof), ('CatBoost', cat_oof)]:
    print(f'  {name:9s} F2@0.5: {f2_score(y, (oof>0.5).astype(int)):.4f} | '
          f'F1-macro: {f1_score(y, (oof>0.5).astype(int), average="macro"):.4f}')
print(f'  {"Ensemble":9s} F2@best: {best_f2:.4f} (thr={best_thr:.3f}) | '
      f'AUC: {roc_auc_score(y, ens_oof):.4f}')
print()
print(classification_report(y, (ens_oof > best_thr).astype(int), target_names=['No', 'Yes']))

In [ ]:
cm = confusion_matrix(y, (ens_oof > best_thr).astype(int), labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes'])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title(f'Ensemble OOF Confusion Matrix (F2={best_f2:.4f})'); plt.show()

## Generate Submission

In [ ]:
test_labels = np.where(ens_test > best_thr, 'Yes', 'No')
submission = sub.copy()
test_label_series = pd.Series(test_labels, index=test.index)
submission.loc[predict_mask, TARGET] = test_label_series[predict_mask].values

print(submission[TARGET].value_counts())
print(f'Null count: {submission[TARGET].isna().sum()}')
submission.to_csv('submission.csv', index=False)
print('\nSaved to submission.csv')
submission.head(10)

In [ ]:
from google.colab import files
files.download('submission.csv')